In [0]:
# Databricks notebook source
# ============================================================
# 1. CARGA DE DATOS Y PARTICIÓN DE VALIDACIÓN
# ============================================================
from pyspark.sql import functions as F
import mlflow
import pandas as pd
import tempfile
import os

from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.functions import vector_to_array

TRAIN_TABLE = "workspace.bigdata_daigreco.tp_final_train"
TEST_TABLE = "workspace.bigdata_daigreco.tp_final_test"

train_df = spark.table(TRAIN_TABLE)
test_df = spark.table(TEST_TABLE)

experiment_name = "/Shared/tp_final_bank_marketing_mlflow"
mlflow.set_experiment(experiment_name)

# Split 80/20 interno sobre Train para Tuning/Validación
fit_df, validation_df = train_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train para tuning: {fit_df.count():,}")
print(f"Validation:        {validation_df.count():,}")
print(f"Test reservado:    {test_df.count():,}")

# COMMAND ----------

# ============================================================
# 2. EVALUADORES Y MÉTRICAS DE CLASE POSITIVA
# ============================================================
roc_evaluator = BinaryClassificationEvaluator(labelCol="DEPOSIT", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
pr_evaluator = BinaryClassificationEvaluator(labelCol="DEPOSIT", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="DEPOSIT", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="DEPOSIT", predictionCol="prediction", metricName="f1")

def positive_class_metrics(predictions):
    tp = predictions.filter((F.col("DEPOSIT") == 1) & (F.col("prediction") == 1)).count()
    fp = predictions.filter((F.col("DEPOSIT") == 0) & (F.col("prediction") == 1)).count()
    fn = predictions.filter((F.col("DEPOSIT") == 1) & (F.col("prediction") == 0)).count()
    tn = predictions.filter((F.col("DEPOSIT") == 0) & (F.col("prediction") == 0)).count()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1_pos = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision_positive": precision,
        "recall_positive": recall,
        "f1_positive": f1_pos
    }

# COMMAND ----------

# ============================================================
# 3. EXPERIMENTACIÓN Y TUNING DE HIPERPARÁMETROS (GBT)
# ============================================================
gbt_configs = [
    {"name": "gbt_depth3", "maxDepth": 3, "maxIter": 50, "stepSize": 0.10},
    {"name": "gbt_baseline", "maxDepth": 5, "maxIter": 50, "stepSize": 0.10},
    {"name": "gbt_iter100", "maxDepth": 5, "maxIter": 100, "stepSize": 0.05},
    {"name": "gbt_depth7", "maxDepth": 7, "maxIter": 50, "stepSize": 0.10}
]

results = []

for config in gbt_configs:
    print(f"Entrenando {config['name']}...")
    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="DEPOSIT",
        maxDepth=config["maxDepth"],
        maxIter=config["maxIter"],
        stepSize=config["stepSize"],
        seed=42
    )

    with mlflow.start_run(run_name=f"tuning_{config['name']}"):
        model = gbt.fit(fit_df)
        preds = model.transform(validation_df)

        roc_auc = roc_evaluator.evaluate(preds)
        pr_auc = pr_evaluator.evaluate(preds)
        acc = accuracy_evaluator.evaluate(preds)
        f1_w = f1_evaluator.evaluate(preds)
        pos_metrics = positive_class_metrics(preds)

        # Log MLflow
        mlflow.log_param("model", "GBTClassifier")
        mlflow.log_param("maxDepth", config["maxDepth"])
        mlflow.log_param("maxIter", config["maxIter"])
        mlflow.log_param("stepSize", config["stepSize"])
        
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("pr_auc", pr_auc)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_weighted", f1_w)
        mlflow.log_metric("precision_positive", pos_metrics["precision_positive"])
        mlflow.log_metric("recall_positive", pos_metrics["recall_positive"])
        mlflow.log_metric("f1_positive", pos_metrics["f1_positive"])

        results.append({
            "model": config["name"],
            "maxDepth": config["maxDepth"],
            "maxIter": config["maxIter"],
            "stepSize": config["stepSize"],
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "accuracy": acc,
            "f1_positive": pos_metrics["f1_positive"]
        })

results_pd = pd.DataFrame(results).sort_values(by="pr_auc", ascending=False).reset_index(drop=True)
display(results_pd)

# COMMAND ----------

# ============================================================
# 4. OPTIMIZACIÓN DEL UMBRAL DE DECISIÓN (THRESHOLD)
# ============================================================
best_gbt = GBTClassifier(
    featuresCol="features",
    labelCol="DEPOSIT",
    maxDepth=3,
    maxIter=50,
    stepSize=0.1,
    seed=42
)

best_gbt_model = best_gbt.fit(fit_df)
best_preds = best_gbt_model.transform(validation_df)

threshold_df = best_preds.withColumn(
    "prob_deposit",
    vector_to_array("probability")[1]
).select("DEPOSIT", "prob_deposit")

thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []

for th in thresholds:
    evaluated = threshold_df.withColumn("prediction_th", (F.col("prob_deposit") >= th).cast("int"))

    tp = evaluated.filter((F.col("DEPOSIT") == 1) & (F.col("prediction_th") == 1)).count()
    fp = evaluated.filter((F.col("DEPOSIT") == 0) & (F.col("prediction_th") == 1)).count()
    fn = evaluated.filter((F.col("DEPOSIT") == 1) & (F.col("prediction_th") == 0)).count()
    tn = evaluated.filter((F.col("DEPOSIT") == 0) & (F.col("prediction_th") == 0)).count()

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0

    threshold_results.append({
        "threshold": th, "precision": prec, "recall": rec, "f1_positive": f1,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn
    })

threshold_results_pd = pd.DataFrame(threshold_results)
display(threshold_results_pd)

best_th_row = threshold_results_pd.loc[threshold_results_pd["f1_positive"].idxmax()]
best_threshold = float(best_th_row["threshold"])
print(f"✅ Umbral óptimo seleccionado por mayor F1+: {best_threshold:.2f}")

# COMMAND ----------

# ============================================================
# 5. EVALUACIÓN FINAL SOBRE CONJUNTO DE TEST RESERVADO
# ============================================================
final_gbt = GBTClassifier(
    featuresCol="features",
    labelCol="DEPOSIT",
    maxDepth=3,
    maxIter=50,
    stepSize=0.1,
    seed=42
)

final_gbt_model = final_gbt.fit(train_df)
final_test_preds = final_gbt_model.transform(test_df)

final_test_preds = final_test_preds.withColumn(
    "prob_deposit",
    vector_to_array("probability")[1]
).withColumn(
    "prediction_final",
    (F.col("prob_deposit") >= best_threshold).cast("int")
)

tp = final_test_preds.filter((F.col("DEPOSIT") == 1) & (F.col("prediction_final") == 1)).count()
fp = final_test_preds.filter((F.col("DEPOSIT") == 0) & (F.col("prediction_final") == 1)).count()
fn = final_test_preds.filter((F.col("DEPOSIT") == 1) & (F.col("prediction_final") == 0)).count()
tn = final_test_preds.filter((F.col("DEPOSIT") == 0) & (F.col("prediction_final") == 0)).count()

precision_final = tp / (tp + fp) if (tp + fp) else 0.0
recall_final = tp / (tp + fn) if (tp + fn) else 0.0
f1_final = (2 * precision_final * recall_final / (precision_final + recall_final)) if (precision_final + recall_final) else 0.0
accuracy_final = (tp + tn) / (tp + tn + fp + fn)

roc_auc_final = roc_evaluator.evaluate(final_test_preds)
pr_auc_final = pr_evaluator.evaluate(final_test_preds)

print("--- RESULTADOS FINALES EN TEST ---")
print(f"ROC-AUC:    {roc_auc_final:.4f}")
print(f"PR-AUC:     {pr_auc_final:.4f}")
print(f"Accuracy:   {accuracy_final:.4f}")
print(f"Precision+: {precision_final:.4f}")
print(f"Recall+:    {recall_final:.4f}")
print(f"F1+:        {f1_final:.4f}")

# COMMAND ----------

# ============================================================
# 6. REGISTRO FINAL DE MÉTRICAS Y ARTEFACTOS EN MLFLOW
# ============================================================
confusion_artifact = pd.DataFrame({
    "actual": [0, 0, 1, 1],
    "predicted": [0, 1, 0, 1],
    "count": [tn, fp, fn, tp]
})

with tempfile.TemporaryDirectory() as tmpdir:
    th_path = os.path.join(tmpdir, "threshold_comparison.csv")
    cm_path = os.path.join(tmpdir, "confusion_matrix_test.csv")

    pd.DataFrame(threshold_results).to_csv(th_path, index=False)
    confusion_artifact.to_csv(cm_path, index=False)

    with mlflow.start_run(run_name="GBT_FINAL_BANK_MARKETING"):
        mlflow.log_param("model", "GBTClassifier")
        mlflow.log_param("threshold", best_threshold)
        mlflow.log_param("maxDepth", 3)
        mlflow.log_param("maxIter", 50)
        mlflow.log_param("stepSize", 0.1)

        mlflow.log_metric("roc_auc", roc_auc_final)
        mlflow.log_metric("pr_auc", pr_auc_final)
        mlflow.log_metric("accuracy", accuracy_final)
        mlflow.log_metric("precision_positive", precision_final)
        mlflow.log_metric("recall_positive", recall_final)
        mlflow.log_metric("f1_positive", f1_final)
        mlflow.log_metric("true_positives", tp)
        mlflow.log_metric("false_positives", fp)

        mlflow.log_artifact(th_path, artifact_path="evaluation")
        mlflow.log_artifact(cm_path, artifact_path="evaluation")

print("✅ Experimento final y artefactos guardados en MLflow.")

{"ts": "2026-10-30 20:52:55.164", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMxABIAEyJDAxYTE4ZTk2LTI1OTUtN2M2MS04ZDZlLTFlMThhYzE1OTc5NTokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4OGJlOWYxSgsIn4eU1wYQwLLNO1ABWAFgAWiNyuDDhcWjDYgBAA==.", "context": {}}
{"ts": "2026-10-30 20:52:55.164", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMxABIAEyJDAxYTE4ZTk2LTI1OTUtN2M2MS04ZDZlLTFlMThhYzE1OTc5NTokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4OGJlOWYxSgsIn4eU1wYQwLLNO1ABWAFgAWiNyuDDhcWjDYgBAA==.", "context": {}}
{"ts": "2026-10-30 20:52:55.164", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMxABIAEyJDAxYTE4ZTk2LTI1OTUtN2M2MS04ZDZlLTFlMThhYzE1OTc5NTokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4

Train para tuning: 26,271
Validation:        6,632
Test reservado:    8,285
Entrenando gbt_depth3...


2026/10/30 20:52:57 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/10/30 20:53:30 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


Entrenando gbt_baseline...


2026/10/30 20:54:12 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


Entrenando gbt_iter100...


... (3 outputs left out)

threshold,precision,recall,f1_positive,TP,FP,FN,TN
0.15,0.4950576606260296,0.8585714285714285,0.6280041797283177,601,613,99,5319
0.2,0.5146252285191956,0.8042857142857143,0.6276477146042362,563,531,137,5401
0.25,0.5342601787487586,0.7685714285714286,0.6303456356180432,538,469,162,5463
0.3,0.5489773950484392,0.7285714285714285,0.6261510128913443,510,419,190,5513
0.35,0.5652173913043478,0.6871428571428572,0.6202450032237267,481,370,219,5562
0.4,0.5970350404312669,0.6328571428571429,0.6144244105409153,443,299,257,5633
0.45,0.6194267515923567,0.5557142857142857,0.5858433734939759,389,239,311,5693
0.5,0.6472795497185742,0.4928571428571429,0.559610705596107,345,188,355,5744


✅ Umbral óptimo seleccionado por mayor F1+: 0.25


2026/10/30 20:58:00 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


--- RESULTADOS FINALES EN TEST ---
ROC-AUC:    0.9391
PR-AUC:     0.6292
Accuracy:   0.8893
Precision+: 0.5039
Recall+:    0.7662
F1+:        0.6080
✅ Experimento final y artefactos guardados en MLflow.
